In [ ]:
# crosswalk
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="mRQRkRQBDCgJRCWwsUYU")
project = rf.workspace("object-project").project("crosswalk-lqgin")
version = project.version(1)
dataset = version.download("yolov8")


In [2]:
import os
import yaml

# Path to your dataset folder (adjust if different)
dataset_dir = "/content/crosswalk-1"

# Function to count images in a split
def count_images(split):
    img_dir = os.path.join(dataset_dir, split, "images")
    if os.path.exists(img_dir):
        return len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    else:
        return 0

# Count images in each split
train_count = count_images("train")
valid_count = count_images("valid")
test_count = count_images("test")  # Some datasets may not have a test set

print(f"Train images: {train_count}")
print(f"Validation images: {valid_count}")
print(f"Test images: {test_count}")

# Load classes from data.yaml
data_yaml_path = os.path.join(dataset_dir, "data.yaml")
with open(data_yaml_path, 'r') as f:
    data_yaml = yaml.safe_load(f)

class_names = data_yaml.get("names", [])
print("\nClasses in dataset:")
for i, cls in enumerate(class_names):
    print(f"{i}: {cls}")



Train images: 8394
Validation images: 799
Test images: 400

Classes in dataset:
0: crosswalk
1: light_red
2: light_green


In [8]:
import os
import cv2
import yaml
import numpy as np
from PIL import Image

# Setup
dataset_dir = "/content/crosswalk-1"
output_dir = "/content/reclassified_labels"
os.makedirs(output_dir, exist_ok=True)

# Load class names
with open(os.path.join(dataset_dir, "data.yaml"), "r") as f:
    data_yaml = yaml.safe_load(f)
class_names = data_yaml.get("names", [])

# New class mapping
new_classes = ["crosswalk", "red light", "green light"]

# Write new YAML
with open(os.path.join(output_dir, "data.yaml"), "w") as f:
    yaml.dump({"names": new_classes, "nc": len(new_classes)}, f)

def detect_light_color(cropped_img):
    """Classify red/green using HSV color mask."""
    hsv = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2HSV)

    red1 = cv2.inRange(hsv, (0, 100, 100), (10, 255, 255))
    red2 = cv2.inRange(hsv, (160, 100, 100), (179, 255, 255))
    green = cv2.inRange(hsv, (40, 40, 40), (90, 255, 255))

    red_pixels = cv2.countNonZero(red1) + cv2.countNonZero(red2)
    green_pixels = cv2.countNonZero(green)

    return 1 if red_pixels > green_pixels else 2  # 1 = red light, 2 = green light

def process_split(split):
    img_dir = os.path.join(dataset_dir, split, "images")
    lbl_dir = os.path.join(dataset_dir, split, "labels")
    new_lbl_dir = os.path.join(output_dir, split, "labels")
    os.makedirs(new_lbl_dir, exist_ok=True)

    for label_file in os.listdir(lbl_dir):
        label_path = os.path.join(lbl_dir, label_file)
        image_path = os.path.join(img_dir, label_file.replace(".txt", ".jpg"))

        if not os.path.exists(image_path):
            continue

        img = cv2.imread(image_path)
        h, w, _ = img.shape
        new_lines = []

        with open(label_path, "r") as f:
            for line in f.readlines():
                parts = line.strip().split()
                cls_id, x, y, bw, bh = map(float, parts)
                if class_names[int(cls_id)] == "light":
                    xmin = int((x - bw / 2) * w)
                    ymin = int((y - bh / 2) * h)
                    xmax = int((x + bw / 2) * w)
                    ymax = int((y + bh / 2) * h)
                    cropped = img[ymin:ymax, xmin:xmax]
                    if cropped.shape[0] < 5 or cropped.shape[1] < 5:
                        continue  # skip tiny boxes
                    new_cls = detect_light_color(cropped)
                    new_lines.append(f"{new_cls} {x} {y} {bw} {bh}")
                else:
                    # Keep other objects as-is (e.g., crosswalk)
                    new_lines.append(line.strip())

        # Write new label file
        with open(os.path.join(new_lbl_dir, label_file), "w") as f_out:
            f_out.write("\n".join(new_lines))

# Process all splits
for split in ["train", "valid", "test"]:
    process_split(split)


KeyboardInterrupt: 

In [12]:
import os
import cv2
import yaml
import numpy as np

# Setup
dataset_dir = "/content/crosswalk-1"
output_dir = "/content/debug_labels"
crop_debug_dir = "/content/debug_crops"
os.makedirs(output_dir, exist_ok=True)

# Load class names
with open(os.path.join(dataset_dir, "data.yaml"), "r") as f:
    data_yaml = yaml.safe_load(f)
class_names = data_yaml.get("names", [])

# New class mapping
new_classes = ["crosswalk", "red light", "green light"]
with open(os.path.join(output_dir, "data.yaml"), "w") as f:
    yaml.dump({"names": new_classes, "nc": len(new_classes)}, f)

def detect_light_color(cropped_img):
    """Classify red/green using HSV color mask."""
    hsv = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2HSV)
    red1 = cv2.inRange(hsv, (0, 100, 100), (10, 255, 255))
    red2 = cv2.inRange(hsv, (160, 100, 100), (179, 255, 255))
    green = cv2.inRange(hsv, (40, 40, 40), (90, 255, 255))
    red_pixels = cv2.countNonZero(red1) + cv2.countNonZero(red2)
    green_pixels = cv2.countNonZero(green)
    return 1 if red_pixels > green_pixels else 2

def process_split(split, limit=10):
    img_dir = os.path.join(dataset_dir, split, "images")
    lbl_dir = os.path.join(dataset_dir, split, "labels")
    new_lbl_dir = os.path.join(output_dir, split, "labels")
    os.makedirs(new_lbl_dir, exist_ok=True)
    os.makedirs(os.path.join(crop_debug_dir, split), exist_ok=True)

    # 🔍 Scan all label files and collect those that contain "light"
    light_label_files = []
    for label_file in sorted(os.listdir(lbl_dir)):
        label_path = os.path.join(lbl_dir, label_file)
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id = int(parts[0])
                    if class_names[cls_id] == "light":
                        light_label_files.append(label_file)
                        break

    print(f"✅ Found {len(light_label_files)} images with 'light' class in {split} split.")

    # Limit to first N
    light_label_files = light_label_files[:limit]

    for label_file in light_label_files:
        base_name = label_file.replace(".txt", "")
        image_path = None
        for ext in [".jpg", ".png", ".jpeg"]:
            candidate = os.path.join(img_dir, base_name + ext)
            if os.path.exists(candidate):
                image_path = candidate
                break
        if image_path is None:
            print(f"⚠️ No image found for {label_file}")
            continue

        img = cv2.imread(image_path)
        h, w, _ = img.shape
        new_lines = []
        label_path = os.path.join(lbl_dir, label_file)

        with open(label_path, "r") as f:
            for i, line in enumerate(f.readlines()):
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls_id, x, y, bw, bh = map(float, parts)
                if class_names[int(cls_id)] == "light":
                    xmin = int((x - bw / 2) * w)
                    ymin = int((y - bh / 2) * h)
                    xmax = int((x + bw / 2) * w)
                    ymax = int((y + bh / 2) * h)
                    cropped = img[ymin:ymax, xmin:xmax]
                    if cropped.shape[0] < 5 or cropped.shape[1] < 5:
                        print(f"⚠️ Skipped small crop in {label_file}")
                        continue
                    new_cls = detect_light_color(cropped)
                    new_lines.append(f"{new_cls} {x} {y} {bw} {bh}")
                    save_path = os.path.join(crop_debug_dir, split, f"{label_file}_{i}_{new_classes[new_cls].replace(' ', '_')}.jpg")
                    cv2.imwrite(save_path, cropped)
                    print(f"{label_file} box {i}: Detected {new_classes[new_cls]}")
                else:
                    new_lines.append(line.strip())

        with open(os.path.join(new_lbl_dir, label_file), "w") as f_out:
            f_out.write("\n".join(new_lines))

# 🔁 Try again on all splits
for split in ["train", "valid", "test"]:
    process_split(split, limit=10)


✅ Found 0 images with 'light' class in train split.
✅ Found 0 images with 'light' class in valid split.
✅ Found 0 images with 'light' class in test split.


In [13]:
!cat /content/crosswalk-1/data.yaml


path: /content/crosswalk-1
train: train/images
val: valid/images
test: test/images
names: ['crosswalk', 'light_red', 'light_green']
